In [1]:
import pandas as pd
import sys
import os

sys.path.append(os.path.join(os.getcwd(), 'config'))

from db_config import get_engine

In [2]:
engine = get_engine()
conn = engine.connect()
print("Connexion OK")

Connexion OK


In [3]:
df = pd.read_sql("SELECT * FROM dbo.netflix_raw", engine)
print(df.shape)
df.head()

(8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,None,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,None,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",None,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,None,None,None,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,None,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [4]:
# Remplacer les NULL textuels
df.replace('NULL', pd.NA, inplace=True)

In [6]:
# Nettoyer les espaces
df['title'] = df['title'].str.strip()
df['director'] = df['director'].str.strip()
df['country'] = df['country'].str.strip()

In [7]:
# Convertir date_added en DATE
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')

In [8]:
# Extraire année, mois depuis date_added
df['added_year']  = df['date_added'].dt.year
df['added_month'] = df['date_added'].dt.month

In [9]:
df.dtypes

show_id                 object
type                    object
title                   object
director                object
cast                    object
country                 object
date_added      datetime64[ns]
release_year             int64
rating                  object
duration                object
listed_in               object
description             object
added_year             float64
added_month            float64
dtype: object

In [10]:
# "90 min" → Movies && "2 Seasons" → TV Shows
df['duration_int']  = df['duration'].str.extract(r'(\d+)').astype(float)
df['duration_unit'] = df['duration'].str.extract(r'([a-zA-Z]+)')

In [11]:
df[['type', 'duration', 'duration_int', 'duration_unit']].sample(5)

,type,duration,duration_int,duration_unit
1546,TV Show,2 Seasons,2.0,Seasons
225,TV Show,1 Season,1.0,Season
2257,Movie,96 min,96.0,min
7373,Movie,78 min,78.0,min
1780,Movie,116 min,116.0,min


In [ ]:
# dimension title
dim_title = df[[
    'show_id', 'type', 'title', 'director',
    'date_added', 'added_year', 'added_month',
    'release_year', 'rating',
    'duration_int', 'duration_unit', 'description'
]].copy()

dim_title.to_sql('dim_title', engine, schema='silver',
                 if_exists='replace', index=False)
print(f"dim_title chargée : {len(dim_title)} lignes")

dim_title chargée : 8807 lignes


In [14]:
# dimension genre
dim_genre = df[['show_id', 'listed_in']].copy()

# Éclater : "Documentaries, International Movies" → 2 lignes
dim_genre = dim_genre.assign(
    genre=dim_genre['listed_in'].str.split(',')
).explode('genre')

dim_genre['genre'] = dim_genre['genre'].str.strip()
dim_genre.drop(columns='listed_in', inplace=True)

dim_genre.to_sql('dim_genre', engine, schema='silver',
                 if_exists='replace', index=False)
print(f"dim_genre chargée : {len(dim_genre)} lignes")

dim_genre chargée : 19323 lignes


In [15]:
# dimension country
dim_country = df[['show_id', 'country']].dropna(subset=['country'])

dim_country = dim_country.assign(
    country=dim_country['country'].str.split(',')
).explode('country')

dim_country['country'] = dim_country['country'].str.strip()

dim_country.to_sql('dim_country', engine, schema='silver',
                   if_exists='replace', index=False)
print(f"dim_country chargée : {len(dim_country)} lignes")

dim_country chargée : 10019 lignes


In [16]:
# dimension cast
dim_cast = df[['show_id', 'cast']].dropna(subset=['cast'])

dim_cast = dim_cast.assign(
    actor=dim_cast['cast'].str.split(',')
).explode('actor')

dim_cast['actor'] = dim_cast['actor'].str.strip()
dim_cast.drop(columns='cast', inplace=True)

dim_cast.to_sql('dim_cast', engine, schema='silver',
                if_exists='replace', index=False)
print(f"dim_cast chargée : {len(dim_cast)} lignes")

dim_cast chargée : 64126 lignes


In [17]:
verif = pd.read_sql("""
    select
        (select count(*) FROM silver.dim_title)   AS dim_title,
        (select count(*) FROM silver.dim_genre)   AS dim_genre,
        (select count(*) FROM silver.dim_country) AS dim_country,
        (select count(*) FROM silver.dim_cast)    AS dim_cast
""", engine)

print(verif)

   dim_title  dim_genre  dim_country  dim_cast
0       8807      19323        10019     64126


In [19]:
for table in ['dim_title', 'dim_genre', 'dim_country', 'dim_cast']:
    df_check = pd.read_sql(f"SELECT * FROM silver.{table}", engine)
    null_counts = df_check.isnull().sum()
    null_counts = null_counts[null_counts > 0]
    if not null_counts.empty:
        print(f"\n {table}:")
        print(null_counts)
    else:
        print(f"\n {table} — aucun NULL")


 dim_title:
director         2634
date_added         10
added_year         10
added_month        10
rating              4
duration_int        3
duration_unit       3
dtype: int64

 dim_genre — aucun NULL

 dim_country — aucun NULL

 dim_cast — aucun NULL
